   

## Wake `SCimilarity` Endpoints

**REQUIREMENTS:**

- Authentication: handled automatically by `databricks-sdk` (uses built-in notebook credentials)
- endpoint_names: 
  - `'<prefix>_scimilarity_gene_order_endpoint'`
  - `'<prefix>_scimilarity_get_embedding_endpoint'`
  - `'<prefix>_scimilarity_search_nearest_endpoint'`

- compute: works with serverless

In [0]:
# dbutils.widgets.removeAll()

In [0]:
# ## to update as widgets input
# import os 
# DATABRICKS_TOKEN = dbutils.secrets.get("<SCOPE>","<SECRET_KEY>") ## to update with APP SP/PAT token?
# os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN

# databricks_instance = dbutils.widgets.text("databricks_instance_url", "<databricks_cloud_url>", "databricks_instance")
# databricks_instance = dbutils.widgets.get("databricks_instance_url")
# os.environ["databricks_instance"] = databricks_instance


# ## User to specify in Widgets -- to replace with gwb-app ones?... 
# gene_order_endpoint = dbutils.widgets.text("gene_order_endpoint_name", "<prefix>_scimilarity_gene_order_endpoint", "gene_order_endpoint")
# get_embedding_endpoint = dbutils.widgets.text("get_embedding_endpoint_name", "<prefix>_scimilarity_get_embedding_endpoint", "get_embedding_endpoint")
# search_nearest_endpoint = dbutils.widgets.text("search_nearest_endpoint_name", "<prefix>_scimilarity_search_nearest_endpoint", "search_nearest_endpoint")

In [0]:
dbutils.widgets.removeAll()

### SDK Setup & Configuration

In [0]:
## SDK client — uses built-in notebook credentials (no PAT/secrets needed)
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

## User to specify prefix (e.g. team or project name)
dbutils.widgets.text("endpoint_prefix", "gwb_mmt", "endpoint_prefix")
prefix = dbutils.widgets.get("endpoint_prefix")

## Endpoint names built from prefix + suffix
ENDPOINT_SUFFIXES = [
    "_scimilarity_gene_order_endpoint",
    "_scimilarity_get_embedding_endpoint",
    "_scimilarity_search_nearest_endpoint",
]

dbutils.widgets.text("gene_order_endpoint_name", f"{prefix}_scimilarity_gene_order_endpoint", "gene_order_endpoint")
dbutils.widgets.text("get_embedding_endpoint_name", f"{prefix}_scimilarity_get_embedding_endpoint", "get_embedding_endpoint")
dbutils.widgets.text("search_nearest_endpoint_name", f"{prefix}_scimilarity_search_nearest_endpoint", "search_nearest_endpoint")

In [0]:
scimilarity_endpoints = [
    dbutils.widgets.get("gene_order_endpoint_name"),
    dbutils.widgets.get("get_embedding_endpoint_name"),
    dbutils.widgets.get("search_nearest_endpoint_name")
]

scimilarity_endpoints

### Endpoint Management Functions
Utility functions to **start**, **stop**, and **check readiness** of serving endpoints via the Databricks SDK.

In [0]:
from databricks.sdk import WorkspaceClient


def start_endpoint(w: WorkspaceClient, endpoint_name: str) -> None:
    """Start a scaled-to-zero serving endpoint."""
    try:
        w.api_client.do(
            "POST",
            f"/api/2.0/serving-endpoints/{endpoint_name}/config:start",
        )
        print(f"Successfully started endpoint: {endpoint_name}")
    except Exception as e:
        print(f"Failed to start endpoint: {endpoint_name}, Error: {e}")


def stop_endpoint(w: WorkspaceClient, endpoint_name: str) -> None:
    """Stop a serving endpoint (scale to zero)."""
    try:
        w.api_client.do(
            "POST",
            f"/api/2.0/serving-endpoints/{endpoint_name}/config:stop",
        )
        print(f"Successfully stopped endpoint: {endpoint_name}")
    except Exception as e:
        print(f"Failed to stop endpoint: {endpoint_name}, Error: {e}")


def check_endpoint_status(w: WorkspaceClient, endpoint_name: str) -> str:
    """Check the readiness status of a serving endpoint."""
    try:
        endpoint = w.serving_endpoints.get(endpoint_name)
        status = (
            endpoint.state.ready.value
            if endpoint.state and endpoint.state.ready
            else "Unknown"
        )
        print(f"Endpoint: {endpoint_name}, Status: {status}")
        return status
    except Exception as e:
        print(f"Failed to check status of endpoint: {endpoint_name}, Error: {e}")
        return "Unknown"


## Example USAGE
# for endpoint in endpoints:
    # start_endpoint(w, endpoint)
    # stop_endpoint(w, endpoint)
    # check_endpoint_status(w, endpoint)

In [0]:
# Alias for downstream cells
endpoints = scimilarity_endpoints

In [0]:
import time


def wait_until_endpoints_ready(
    w: WorkspaceClient,
    endpoints: list[str],
    check_interval: int = 60,
) -> None:
    """Block until all serving endpoints report READY status."""
    all_ready = False
    while not all_ready:
        all_ready = True
        for endpoint in endpoints:
            status = check_endpoint_status(w, endpoint)
            if status != "READY":
                all_ready = False
                print(
                    f"Endpoint {endpoint} is not ready yet. "
                    f"Checking again in {check_interval} seconds..."
                )
                break
        if not all_ready:
            time.sleep(check_interval)
    print("All endpoints are ready!")

In [0]:
# 15++ mins

# wait_until_endpoints_ready(databricks_instance, endpoints, DATABRICKS_TOKEN, check_interval=60)

### Sample Inference Payloads
Minimal request payloads for each SCimilarity endpoint, used to send a lightweight "wake-up" call.

In [0]:
import json

gene_order_ep = dbutils.widgets.get("gene_order_endpoint_name")
get_embedding_ep = dbutils.widgets.get("get_embedding_endpoint_name")
search_nearest_ep = dbutils.widgets.get("search_nearest_endpoint_name")

# ── Gene order: simple wake-up payload ──
gene_order_payload = {
    "data": {
        "dataframe_split": {
            "columns": ["input"],
            "data": [["get_gene_order"]]
        }
    }
}

# ── Get embedding: 28,231-feature zero vector (correct shape for wake-up) ──
n_genes = 28_231
celltype_sample_json = json.dumps({
    "columns": ["celltype_subsample"],
    "index": ["0"],
    "data": [[[0.0] * n_genes]]
})
celltype_sample_obs_json = json.dumps({
    "columns": ["disease", "tissue", "celltype_name"],
    "index": ["0"],
    "data": [["healthy", "blood", "T cell"]]
})
get_embedding_payload = {
    "data": {
        "dataframe_split": {
            "columns": ["celltype_sample", "celltype_sample_obs"],
            "data": [[celltype_sample_json, celltype_sample_obs_json]]
        }
    }
}

# ── Search nearest: 128-dim zero embedding ──
search_nearest_payload = {
    "data": {
        "dataframe_split": {
            "columns": ["embedding", "params"],
            "data": [[[0.0] * 128, json.dumps({"k": 10})]]
        }
    }
}

request_dict = {
    gene_order_ep: gene_order_payload,
    get_embedding_ep: get_embedding_payload,
    search_nearest_ep: search_nearest_payload,
}

print(f"Configured {len(request_dict)} endpoint payloads:")
for ep in request_dict:
    print(f"  • {ep}")

In [0]:
request_dict.keys()

In [0]:
# request_dict['<prefix>_scimilarity_get_embedding_endpoint']["data"]

In [0]:
# request_dict['<prefix>_scimilarity_search_nearest_endpoint']['data']

In [0]:
# for endpoint, payload in request_dict.items():
#     print(endpoint)
#     print(payload['data'])

In [0]:
# import requests
# import json

# # Function to wake up the endpoints
# def wake_up_endpoints(databricks_instance, request_dict, token):
#     headers = {
#         "Authorization": f"Bearer {token}",
#         "Content-Type": "application/json"
#     }
    
#     for endpoint, payload in request_dict.items():
#         url = f"https://{databricks_instance}/serving-endpoints/{endpoint}/invocations"
#         try:
#             # Ensure the payload data is JSON serializable
#             json_payload = json.dumps(payload["data"])
#             response = requests.post(
#                 url,
#                 headers=headers,
#                 data=json_payload
#             )
#             if response.status_code == 200:
#                 print(f"Endpoint {endpoint} is awake and responded successfully.")
#             else:
#                 print(f"Failed to wake up the endpoint {endpoint}. Status code: {response.status_code}, Response: {response.text}")
#         except (TypeError, ValueError) as e:
#             print(f"Error serializing JSON for endpoint {endpoint}: {e}")


# # Call the function to wake up the endpoints
# wake_up_endpoints(databricks_instance, request_dict, DATABRICKS_TOKEN)

### Parallel Wake-Up Functions
Send inference requests to all endpoints concurrently using `ThreadPoolExecutor` and the Databricks SDK.

In [0]:
import json
import requests
from concurrent.futures import ThreadPoolExecutor
from databricks.sdk import WorkspaceClient


def send_request(
    w: WorkspaceClient, endpoint: str, payload: dict
) -> None:
    """Send an invocation request to a serving endpoint.

    Uses requests.post() directly to avoid SDK deserialization issues.
    Any HTTP response (even a model error) means the endpoint is awake.
    Only connection/timeout failures indicate the endpoint is still scaling.
    """
    try:
        host = w.config.host.rstrip("/")
        token = w.config.token
        url = f"{host}/serving-endpoints/{endpoint}/invocations"
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }
        response = requests.post(
            url, headers=headers, data=json.dumps(payload["data"])
        )
        if response.status_code == 200:
            print(f"Endpoint {endpoint} is awake and responded successfully.")
        else:
            # A non-200 response still means the container is live
            print(
                f"Endpoint {endpoint} is awake "
                f"(HTTP {response.status_code} — model error on sample payload is expected)."
            )
    except requests.exceptions.ConnectionError:
        print(
            f"Endpoint {endpoint} is not reachable "
            f"(connection error — may still be scaling up)."
        )
    except Exception as e:
        print(f"Failed to wake up endpoint {endpoint}. Error: {e}")


def send_inference_inputs_parallel(
    w: WorkspaceClient, request_dict: dict
) -> None:
    """Send invocation requests to all endpoints in parallel."""
    with ThreadPoolExecutor() as executor:
        futures = [
            executor.submit(send_request, w, endpoint, payload)
            for endpoint, payload in request_dict.items()
        ]
        for future in futures:
            future.result()

### Readiness Check & Keep-Alive Loop

In [0]:
wait_until_endpoints_ready(
    w,
    endpoints=endpoints,
    check_interval=60,
)

In [0]:
# -- Tunables --
num_hours_to_keep_alive = 3
num_mins_to_ping = 15

import time

# Periodically ping endpoints to prevent scale-to-zero
for i in range(num_hours_to_keep_alive * 60 // num_mins_to_ping):
    send_inference_inputs_parallel(w, request_dict)
    print(f"sleeping for {num_mins_to_ping} minutes")
    print("----------------------")
    time.sleep(num_mins_to_ping * 60)